In [1]:
# ============================================
# CELL 1: Install dependencies
# ============================================
import sys
import subprocess
import importlib

REQUIRED_PACKAGES = [
    "qdrant-client>=1.9.0",
    "sentence-transformers>=2.6.0",
    "transformers>=4.44.0",
    "accelerate>=0.33.0",
    "bitsandbytes>=0.43.1",
    "python-dotenv>=1.0.0",
    "pandas>=2.0.0",
]

def ensure_packages(packages):
    for pkg in packages:
        module_name = pkg.split("==")[0].split(">=")[0].replace("-", "_")
        try:
            importlib.import_module(module_name)
        except Exception:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

ensure_packages(REQUIRED_PACKAGES)
print("Packages ready.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 398.1/398.1 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 33.8 MB/s eta 0:00:00
Packages ready.


In [2]:
# ============================================
# CELL 2: Imports and configuration
# ============================================
import os
import re
import json
import importlib
import torch
import pandas as pd
import numpy as np
from typing import List, Dict, Any, Tuple, Optional
from dataclasses import dataclass, field
from enum import Enum
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed

from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient
from qdrant_client.http.models import Filter, FieldCondition, MatchValue, Range
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig


pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 150)

# ============================================
# Configuration helpers
# ============================================
def _read_kaggle_secret(key_name):
    try:
        kaggle_secrets = importlib.import_module("kaggle_secrets")
        user_secrets = kaggle_secrets.UserSecretsClient()
        return user_secrets.get_secret(key_name)
    except Exception:
        return None

def _get_config_value(secret_keys, env_keys=None, default=None):
    for key in secret_keys:
        val = _read_kaggle_secret(key)
        if val not in (None, ""):
            return val, f"secret:{key}"
    env_keys = env_keys or []
    for key in env_keys:
        val = os.getenv(key)
        if val not in (None, ""):
            return val, f"env:{key}"
    return default, "default"

# ============================================
# Load configuration
# ============================================
QDRANT_API_KEY, src_qdrant_key = _get_config_value(
    ["QDRAN_API_KEY", "QDRANT_API_KEY"],
    ["QDRAN_API_KEY", "QDRANT_API_KEY"],
)
QDRANT_URL, src_qdrant_url = _get_config_value(
    ["QDRAN_API_URL", "QDRANT_API_URL", "QDRANT_URL"],
    ["QDRAN_API_URL", "QDRANT_API_URL", "QDRANT_URL"],
)

if not QDRANT_URL:
    raise ValueError("Missing Qdrant URL. Set Kaggle Secret QDRANT_URL")

COLLECTION_PLACES = "places_danang"
COLLECTION_PLACE_REVIEWS = "place_reviews_danang"
COLLECTION_RESTAURANTS = "restaurants_danang"
COLLECTION_RESTAURANT_REVIEWS = "restaurant_reviews_danang"
COLLECTION_ACCOMMODATION_HOTELS = "accommodation_hotels_danang"
COLLECTION_ACCOMMODATION_ROOMS = "accommodation_rooms_danang"
COLLECTION_ACCOMMODATION_REVIEWS = "accommodation_reviews_danang"

EMBED_MODEL_NAME = "BAAI/bge-m3"
LOCAL_LLM_MODEL = "Qwen/Qwen2.5-3B-Instruct"
USE_4BIT = True

print("Qdrant URL:", QDRANT_URL)
print("Collections:", COLLECTION_PLACES, COLLECTION_RESTAURANTS, COLLECTION_ACCOMMODATION_HOTELS)
print("Embedding model:", EMBED_MODEL_NAME)
print("CUDA available:", torch.cuda.is_available())

Qdrant URL: https://511742a9-e9a3-4261-91dd-5dbb998f21a7.us-west-1-0.aws.cloud.qdrant.io:6333
Collections: places_danang restaurants_danang accommodation_hotels_danang
Embedding model: BAAI/bge-m3
CUDA available: True


In [3]:
# ============================================
# CELL 3: Define Query Intent and Data Models (FIXED)
# ============================================

class QueryIntent(Enum):
    """Query intent types for better retrieval"""
    HOTEL_SEARCH = "hotel_search"           # Tìm khách sạn
    RESTAURANT_SEARCH = "restaurant_search" # Tìm nhà hàng
    PLACE_SEARCH = "place_search"           # Tìm địa điểm du lịch
    REVIEW_SEARCH = "review_search"         # Tìm review/đánh giá
    ROOM_SEARCH = "room_search"             # Tìm thông tin phòng
    PRICE_SEARCH = "price_search"           # Tìm thông tin giá
    GENERAL = "general"                     # Truy vấn chung
    SPECIFIC_SEARCH = "specific_search"
    
   
@dataclass
class SearchResult:
    """Structured search result"""
    point_id: str
    collection: str
    score: float
    entity_name: str = ""
    place_name: str = ""
    district: str = ""
    rating: Optional[float] = None
    min_price: Optional[float] = None
    max_price: Optional[float] = None
    address: str = ""
    content: str = ""
    parent_entity_name: Optional[str] = None
    parent_entity_id: Optional[str] = None
    parent_rating: Optional[float] = None  # NEW: for review parent rating
    parent_address: Optional[str] = None   # NEW: for review parent address
    room_name: Optional[str] = None
    capacity: Optional[int] = None
    bed_type: Optional[str] = None
    area_m2: Optional[float] = None
    room_view: Optional[str] = None
    cuisine: Optional[str] = None
    restaurant_type: Optional[str] = None
    check_in_time: Optional[str] = None
    check_out_time: Optional[str] = None
    time_open: Optional[str] = None
    time_close: Optional[str] = None
    tags: Optional[List[str]] = None
    review_count: Optional[int] = None
    star_rating: Optional[float] = None
    price_level: Optional[str] = None
    price_currency: Optional[str] = None
    
    def to_dict(self) -> Dict:
        return {k: v for k, v in self.__dict__.items() if v is not None}
    
    def get_display_name(self) -> str:
        """Get best available name for display - FIXED for reviews"""
        # For reviews, ALWAYS use parent entity name first
        if self.collection in [
            COLLECTION_ACCOMMODATION_REVIEWS, 
            COLLECTION_RESTAURANT_REVIEWS, 
            COLLECTION_PLACE_REVIEWS
        ]:
            if self.parent_entity_name and self.parent_entity_name not in ["None", "", "null"]:
                return self.parent_entity_name
            if self.entity_name and self.entity_name not in ["None", "", "null"]:
                return self.entity_name
            if self.place_name and self.place_name not in ["None", "", "null"]:
                return self.place_name
            return "Đang cập nhật"
        
        # For entities
        if self.entity_name and self.entity_name not in ["None", "", "null"]:
            return self.entity_name
        if self.place_name and self.place_name not in ["None", "", "null"]:
            return self.place_name
        return "Unknown"
    
    def get_price_display(self) -> str:
        """Format price for display"""
        if self.min_price is None:
            return "Không có thông tin giá"
        if self.max_price and self.max_price > self.min_price:
            return f"{self.min_price:,.0f} - {self.max_price:,.0f} VND"
        return f"{self.min_price:,.0f} VND"
    
    def get_rating_display(self) -> str:
        """Format rating for display - FIXED for reviews"""
        # For reviews, use parent rating if available
        rating_value = self.parent_rating if self.parent_rating else self.rating
        
        if rating_value is None:
            return "Chưa có đánh giá"
        
        if self.review_count and self.review_count > 0:
            return f"{rating_value:.1f}/10 ({self.review_count:,} đánh giá)"
        return f"{rating_value:.1f}/10"
    
    def get_address_display(self) -> str:
        """Get address for display - FIXED for reviews"""
        # For reviews, use parent address if available
        if self.collection in [
            COLLECTION_ACCOMMODATION_REVIEWS, 
            COLLECTION_RESTAURANT_REVIEWS, 
            COLLECTION_PLACE_REVIEWS
        ]:
            if self.parent_address and self.parent_address not in ["None", "", "null"]:
                return self.parent_address
        return self.address if self.address not in ["None", "", "null"] else "Chưa có địa chỉ"


class CollectionRegistry:
    """Registry for all collections with metadata"""
    
    COLLECTIONS = {
        COLLECTION_PLACES: {
            "type": "place",
            "weight": 1.2,
            "display_name": "Địa điểm du lịch",
            "priority_for_intent": {
                QueryIntent.PLACE_SEARCH: 1.5,
                QueryIntent.GENERAL: 1.2,
            }
        },
        COLLECTION_RESTAURANTS: {
            "type": "restaurant",
            "weight": 1.2,
            "display_name": "Nhà hàng",
            "priority_for_intent": {
                QueryIntent.RESTAURANT_SEARCH: 1.5,
                QueryIntent.GENERAL: 1.2,
            }
        },
        COLLECTION_ACCOMMODATION_HOTELS: {
            "type": "hotel",
            "weight": 1.3,
            "display_name": "Khách sạn",
            "priority_for_intent": {
                QueryIntent.HOTEL_SEARCH: 1.6,
                QueryIntent.ROOM_SEARCH: 1.4,
                QueryIntent.PRICE_SEARCH: 1.3,
                QueryIntent.GENERAL: 1.3,
            }
        },
        COLLECTION_PLACE_REVIEWS: {
            "type": "review",
            "weight": 1.0,
            "display_name": "Đánh giá địa điểm",
            "priority_for_intent": {
                QueryIntent.REVIEW_SEARCH: 1.6,
            }
        },
        COLLECTION_RESTAURANT_REVIEWS: {
            "type": "review",
            "weight": 1.0,
            "display_name": "Đánh giá nhà hàng",
            "priority_for_intent": {
                QueryIntent.REVIEW_SEARCH: 1.6,
            }
        },
        COLLECTION_ACCOMMODATION_REVIEWS: {
            "type": "review",
            "weight": 1.0,
            "display_name": "Đánh giá khách sạn",
            "priority_for_intent": {
                QueryIntent.REVIEW_SEARCH: 1.6,
            }
        },
        COLLECTION_ACCOMMODATION_ROOMS: {
            "type": "room",
            "weight": 0.95,
            "display_name": "Thông tin phòng",
            "priority_for_intent": {
                QueryIntent.ROOM_SEARCH: 1.8,
            }
        },
    }
    
    @classmethod
    def get_collections_by_intent(cls, intent: QueryIntent) -> List[str]:
        """Get relevant collections for an intent - FIXED to avoid mixing"""
        if intent == QueryIntent.HOTEL_SEARCH:
            # ONLY hotels, no restaurants
            return [COLLECTION_ACCOMMODATION_HOTELS, COLLECTION_ACCOMMODATION_REVIEWS, COLLECTION_ACCOMMODATION_ROOMS]
        elif intent == QueryIntent.RESTAURANT_SEARCH:
            # ONLY restaurants
            return [COLLECTION_RESTAURANTS, COLLECTION_RESTAURANT_REVIEWS]
        elif intent == QueryIntent.PLACE_SEARCH:
            # ONLY places
            return [COLLECTION_PLACES, COLLECTION_PLACE_REVIEWS]
        elif intent == QueryIntent.REVIEW_SEARCH:
            return [COLLECTION_PLACE_REVIEWS, COLLECTION_RESTAURANT_REVIEWS, COLLECTION_ACCOMMODATION_REVIEWS]
        elif intent == QueryIntent.ROOM_SEARCH:
            return [COLLECTION_ACCOMMODATION_ROOMS, COLLECTION_ACCOMMODATION_HOTELS, COLLECTION_ACCOMMODATION_REVIEWS]
        elif intent == QueryIntent.PRICE_SEARCH:
            return [COLLECTION_ACCOMMODATION_HOTELS, COLLECTION_RESTAURANTS]
        else:
            return [COLLECTION_ACCOMMODATION_HOTELS, COLLECTION_RESTAURANTS, COLLECTION_PLACES]
    
    @classmethod
    def get_weight(cls, collection: str, intent: QueryIntent) -> float:
        """Get weight for a collection based on intent"""
        config = cls.COLLECTIONS.get(collection, {"weight": 1.0})
        base_weight = config["weight"]
        priority = config.get("priority_for_intent", {}).get(intent, 1.0)
        return base_weight * priority

In [4]:
# ============================================
# CELL 4: Qdrant client and retrieval functions (FIXED)
# ============================================

# Connect to Qdrant
client = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY, timeout=60)

# Load embedding model
encoder = SentenceTransformer(EMBED_MODEL_NAME, device="cuda" if torch.cuda.is_available() else "cpu")

# ===== THÊM VÀO: Cache và fetch parent function =====
_parent_cache = {}

def fetch_parent_entity(parent_id: str, collection_name: str) -> Optional[Dict]:
    """Fetch parent entity by ID with caching"""
    if not parent_id or parent_id in ["None", "", "null"]:
        return None
    
    if parent_id in _parent_cache:
        return _parent_cache[parent_id]
    
    # Determine which collection to query based on review type
    if "accommodation" in collection_name or "hotel" in collection_name:
        parent_collection = COLLECTION_ACCOMMODATION_HOTELS
    elif "restaurant" in collection_name:
        parent_collection = COLLECTION_RESTAURANTS
    elif "place" in collection_name:
        parent_collection = COLLECTION_PLACES
    else:
        return None
    
    try:
        points = client.retrieve(
            collection_name=parent_collection,
            ids=[parent_id],
            with_payload=True,
        )
        
        if points:
            payload = points[0].payload or {}
            result = {
                "name": payload.get("entity_name") or payload.get("place_name"),
                "rating": payload.get("rating"),
                "address": payload.get("address"),
                "district": payload.get("district"),
            }
            _parent_cache[parent_id] = result
            return result
    except Exception as e:
        print(f"Warning: Cannot fetch parent {parent_id}: {e}")
    
    return None


def extract_payload(point) -> Dict:
    """Extract payload from Qdrant point"""
    if hasattr(point, "payload"):
        return point.payload or {}
    if isinstance(point, dict):
        return point.get("payload", {})
    return {}

def extract_score(point) -> float:
    """Extract score from Qdrant point"""
    if hasattr(point, "score"):
        return point.score
    if isinstance(point, dict):
        return point.get("score", 0.0)
    return 0.0

def retrieve_from_collection(
    collection_name: str,
    query_vector: List[float],
    top_k: int = 5,
    filters: Optional[Dict] = None,
    score_threshold: float = 0.3
) -> List[SearchResult]:
    """Retrieve from a single collection with filters"""
    try:
       
        # Build filter conditions
        conditions = []
        if filters:
            if filters.get("district"):
                conditions.append(
                    FieldCondition(
                        key="district",
                        match=MatchValue(value=str(filters["district"]).lower().strip())
                    )
                )
            if filters.get("min_rating"):
                conditions.append(
                    FieldCondition(
                        key="rating",
                        range=Range(gte=float(filters["min_rating"]))
                    )
                )
            if filters.get("max_price"):
                conditions.append(
                    FieldCondition(
                        key="min_price_vnd",
                        range=Range(lte=float(filters["max_price"]))
                    )
                )
                print(f"  [DEBUG] {collection_name}: đã thêm condition max_price <= {filters['max_price']}")
            if filters.get("min_price"):
                conditions.append(
                    FieldCondition(
                        key="min_price_vnd",
                        range=Range(gte=float(filters["min_price"]))
                    )
                )
        
        # Chỉ tạo filter nếu có conditions
        q_filter = Filter(must=conditions) if conditions else None
        
      
        # Query Qdrant
        result = client.query_points(
            collection_name=collection_name,
            query=query_vector,
            query_filter=q_filter,
            limit=top_k,
            with_payload=True,
            with_vectors=False,
        )
        
        points = result.points if hasattr(result, "points") else result
        
     
        
        # Convert to SearchResult objects
        results = []
        price_filtered_count = 0
        
        for point in points:
            score = extract_score(point)
            if score < score_threshold:
                continue
                
            payload = extract_payload(point)
            
            # Lọc thủ công (Đã tối ưu hóa logic khoảng giá)
            skip = False
            if filters:
                min_price = payload.get("min_price_vnd")
                max_price = payload.get("max_price_vnd")
                
                # CHẶN TRẦN GIÁ (max_price)
                if filters.get("max_price"):
                    # Nếu giá thấp nhất của quán còn cao hơn cả mức khách chịu chi -> LOẠI thẳng
                    if min_price and min_price > filters["max_price"]:
                        skip = True
                        print(f"  [DEBUG] {collection_name}: LOẠI {payload.get('entity_name') or payload.get('place_name')} - giá thấp nhất {min_price:,.0f} > trần {filters['max_price']:,.0f}")
                    
                    # [NÂNG CẤP AN TOÀN]: Nếu quán có giá tối đa vượt quá 1.5 lần mức khách muốn, 
                    # cũng loại luôn để tránh giới thiệu những nơi quá đắt đỏ so với túi tiền của họ.
                    elif max_price and max_price > (filters["max_price"] * 1.5):
                        skip = True
                        print(f"  [DEBUG] {collection_name}: LOẠI {payload.get('entity_name') or payload.get('place_name')} - giá cao nhất {max_price:,.0f} quá cao so với trần {filters['max_price']:,.0f}")
                        
                # CHẶN SÀN GIÁ (min_price)
                if filters.get("min_price") and not skip:
                    # Nếu có giá cao nhất mà giá cao nhất lại thấp hơn mức sàn khách yêu cầu -> LOẠI
                    if max_price and max_price < filters["min_price"]:
                        skip = True
                    # Hoặc nếu không có giá cao nhất, nhưng giá thấp nhất thấp hơn mức sàn -> LOẠI
                    elif min_price and min_price < filters["min_price"]:
                        skip = True
            
            if skip:
                price_filtered_count += 1
                continue
            
            result_obj = SearchResult(
                point_id=str(point.id) if hasattr(point, "id") else str(payload.get("id")),
                collection=collection_name,
                score=score,
                entity_name=payload.get("entity_name", ""),
                place_name=payload.get("place_name", ""),
                district=payload.get("district", ""),
                rating=float(payload.get("rating")) if payload.get("rating") else None,
                min_price=float(payload.get("min_price_vnd")) if payload.get("min_price_vnd") else None,
                max_price=float(payload.get("max_price_vnd")) if payload.get("max_price_vnd") else None,
                address=payload.get("address", ""),
                content=payload.get("content", ""),
                parent_entity_name=payload.get("parent_entity_name"),
                parent_entity_id=payload.get("parent_entity_id"),
                parent_rating=None,
                parent_address=None,
                room_name=payload.get("room_name"),
                capacity=int(payload.get("capacity")) if payload.get("capacity") else None,
                bed_type=payload.get("bed_type"),
                area_m2=float(payload.get("area_m2")) if payload.get("area_m2") else None,
                room_view=payload.get("room_view"),
                cuisine=payload.get("cuisine"),
                restaurant_type=payload.get("restaurant_type"),
                check_in_time=payload.get("check_in_time"),
                check_out_time=payload.get("check_out_time"),
                time_open=payload.get("time_open"),
                time_close=payload.get("time_close"),
                tags=payload.get("tags"),
                review_count=int(payload.get("review_count")) if payload.get("review_count") else None,
                star_rating=float(payload.get("star_rating")) if payload.get("star_rating") else None,
                price_level=payload.get("price_level"),
                price_currency=payload.get("price_currency"),
            )
            
            # For review collections, fetch parent entity info
            if collection_name in [COLLECTION_ACCOMMODATION_REVIEWS, COLLECTION_RESTAURANT_REVIEWS, COLLECTION_PLACE_REVIEWS]:
                parent_id = result_obj.parent_entity_id
                if parent_id and parent_id not in ["None", "", "null"]:
                    parent = fetch_parent_entity(parent_id, collection_name)
                    if parent:
                        if parent.get("name") and parent.get("name") not in ["None", "", "null"]:
                            result_obj.parent_entity_name = parent.get("name")
                        if parent.get("rating"):
                            result_obj.parent_rating = parent.get("rating")
                        if parent.get("address"):
                            result_obj.parent_address = parent.get("address")
                        if parent.get("district") and not result_obj.district:
                            result_obj.district = parent.get("district")
            
            results.append(result_obj)
        
        
        return results
        
    except Exception as e:
        print(f"Warning: Error retrieving from {collection_name}: {e}")
        return []


def retrieve_by_intent(
    query: str,
    intent: Optional[QueryIntent] = None,
    top_k_per_collection: int = 5,
    max_total_results: int = 20,
    filters: Optional[Dict] = None
) -> List[SearchResult]:
    """Main retrieval function with intent-based routing"""
    
    # [ĐÃ SỬA] Fallback an toàn: Nếu quên truyền intent, mặc định tìm kiếm toàn bộ
    if intent is None:
        intent = QueryIntent.GENERAL
    
    # Get relevant collections
    collections = CollectionRegistry.get_collections_by_intent(intent)
    
    # Encode query
    query_vector = encoder.encode([query], normalize_embeddings=True)[0].tolist()
    
    # Retrieve from all relevant collections in parallel
    all_results = []
    
    def retrieve_single(collection):
        return retrieve_from_collection(
            collection_name=collection,
            query_vector=query_vector,
            top_k=top_k_per_collection,
            filters=filters
        )
    
    with ThreadPoolExecutor(max_workers=min(4, len(collections))) as executor:
        future_to_collection = {
            executor.submit(retrieve_single, collection): collection
            for collection in collections
        }
        for future in as_completed(future_to_collection):
            try:
                results = future.result()
                all_results.extend(results)
            except Exception as e:
                print(f"Error retrieving: {e}")
    
    # Apply reranking
    ranked_results = rerank_results(all_results, query, intent)
    
    # Lọc lại theo giá một lần nữa
    if filters and filters.get("max_price"):
        before_filter = len(ranked_results)
        ranked_results = [r for r in ranked_results if r.min_price and r.min_price <= filters["max_price"]]
    
    # Return top results
    return ranked_results[:max_total_results]
    def retrieve_single(collection):
        return retrieve_from_collection(
            collection_name=collection,
            query_vector=query_vector,
            top_k=top_k_per_collection,
            filters=filters  # ← QUAN TRỌNG: phải truyền filters
        )
    
    with ThreadPoolExecutor(max_workers=min(4, len(collections))) as executor:
        future_to_collection = {
            executor.submit(retrieve_single, collection): collection
            for collection in collections
        }
        for future in as_completed(future_to_collection):
            try:
                results = future.result()
                all_results.extend(results)
            except Exception as e:
                print(f"Error retrieving: {e}")
    
    # Apply reranking
    ranked_results = rerank_results(all_results, query, intent)
    
    # Lọc lại theo giá một lần nữa
    if filters and filters.get("max_price"):
        before_filter = len(ranked_results)
        ranked_results = [r for r in ranked_results if r.min_price and r.min_price <= filters["max_price"]]
    
    # Return top results
    return ranked_results[:max_total_results]

    def deduplicate_results(results: List[SearchResult]) -> List[SearchResult]:
        """Gộp các thực thể trùng tên, giữ lại thực thể có đầy đủ thông tin hoặc điểm cao hơn"""
        seen_names = set()
        unique_results = []
        
        for r in results:
            name = r.get_display_name().lower().strip()
            if name not in seen_names:
                seen_names.add(name)
                unique_results.append(r)
            else:
                # Nếu trùng tên, có thể cập nhật thêm thông tin bổ sung (nếu cần)
                pass
                
        return unique_results

def rerank_results(
    results: List[SearchResult],
    query: str,
    intent: QueryIntent
) -> List[SearchResult]:
    """Rerank results using multiple signals and strict keyword penalties"""
    
    query_lower = query.lower()
    
    # Định nghĩa các cặp từ khóa nhạy cảm để tránh hiện tượng "râu ông nọ cắm cằm bà kia"
    mismatch_rules = {
        "hải sản": ["gà rán", "burger", "lotteria", "jollibee", "pizza", "chè", "cafe", "coffee"],
        "mì quảng": ["pizza", "cinema", "bar", "lounge", "buffet", "chè", "cafe", "coffee"],
        "đồ nướng": ["bánh sầu riêng", "quán chay", "chè", "trà sữa", "cafe", "coffee"],
        "khách sạn": ["homestay", "hostel", "dorm"] # Nếu khách tìm khách sạn xịn, hạ bớt dorm/hostel
    }
    
    for result in results:
        final_score = result.score
        entity_name_lower = result.get_display_name().lower()
        
        # 1. Áp dụng trọng số Collection dựa trên Intent
        collection_weight = CollectionRegistry.get_weight(result.collection, intent)
        final_score *= collection_weight
        
        # 2. Boost điểm cho Quận/Huyện trùng khớp (Tăng mạnh từ 0.15 lên 0.3)
        if result.district and result.district.lower() in query_lower:
            final_score += 0.3
            
        # 3. Boost điểm cho thực thể có Rating chất lượng (Dựa trên số lượng review)
        rating_value = result.parent_rating if result.parent_rating else result.rating
        review_count = result.review_count or 0
        if rating_value is not None:
            if review_count > 5:
                final_score += min(0.4, rating_value / 20)
            else:
                final_score += min(0.15, rating_value / 40) # Ít review thì boost ít tránh điểm ảo
                
        # 4. CƠ CHẾ PHẠT (PENALTY) ĐỂ KHỬ NHIỄU VECTOR SEARCH
        for key, bad_words in mismatch_rules.items():
            if key in query_lower:
                # Nếu câu hỏi chứa từ khóa cốt lõi (vd: hải sản) nhưng tên thực thể chứa từ lạc quẻ (vd: lotteria)
                if any(word in entity_name_lower for word in bad_words):
                    final_score -= 0.8 # Trừ điểm cực nặng để đẩy xuống đáy danh sách
                    print(f"  [DEBUG RERANK] Hạ điểm phạt thực thể lệch ngành: {result.get_display_name()}")

        # 5. Tối ưu cho nhu cầu tiếp đối tác / sang trọng
        if "đối tác" in query_lower or "tiếp khách" in query_lower or "sang trọng" in query_lower:
            if rating_value and rating_value < 7.5:
                final_score -= 0.3
            if "buffet" in entity_name_lower:
                final_score -= 0.15
                
        result.score = final_score
        
    # Sắp xếp lại danh sách sau khi đã xử lý tất cả các chiều thông tin
    results.sort(key=lambda x: x.score, reverse=True)
    
    return results

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

In [5]:
# ============================================
# CELL 5: LLM Setup
# ============================================

# Load LLM
if torch.cuda.is_available() and USE_4BIT:
    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.float16,
    )
else:
    quant_config = None

tokenizer = AutoTokenizer.from_pretrained(LOCAL_LLM_MODEL, trust_remote_code=True)

model_kwargs = {
    "trust_remote_code": True,
    "device_map": "auto",
}

if torch.cuda.is_available():
    model_kwargs["dtype"] = torch.float16

if quant_config is not None:
    model_kwargs["quantization_config"] = quant_config

model = AutoModelForCausalLM.from_pretrained(LOCAL_LLM_MODEL, **model_kwargs)
model.eval()

if tokenizer.pad_token is None and tokenizer.eos_token is not None:
    tokenizer.pad_token = tokenizer.eos_token

print("Local LLM ready:", LOCAL_LLM_MODEL)


def generate_answer(
    query: str,
    results: List[SearchResult],
    max_new_tokens: int = 512,
    temperature: float = 0.2,
    intent: Optional[QueryIntent] = None # <-- THÊM THAM SỐ INTENT
) -> str:
    """Generate answer using LLM with retrieved context (Optimized for Specific and General Search)"""
    
    # ----------------------------------------------------
    # BƯỚC 1: LỌC TRÙNG LẶP THỰC THỂ (Deduplication)
    # ----------------------------------------------------
    unique_results = []
    seen_names = set()
    for r in results:
        name_lower = r.get_display_name().lower().strip()
        if name_lower not in seen_names:
            seen_names.add(name_lower)
            unique_results.append(r)
    
    # ----------------------------------------------------
    # BƯỚC 2: FORMAT CONTEXT GỬI CHO LLM
    # ----------------------------------------------------
    context_parts = []
    for i, r in enumerate(unique_results[:8], 1): # Sử dụng danh sách đã lọc trùng
        context = f"""
[{i}] {r.get_display_name()}
   - Loại: {r.collection}
   - Quận/Huyện: {r.district or 'Chưa có'}
   - Đánh giá: {r.get_rating_display()}
   - Giá: {r.get_price_display()}
   - Địa chỉ: {r.address or 'Chưa có'}"""
        
        if r.content:
            context += f"\n   - Nội dung/Mô tả: {r.content[:300]}..."
        if r.room_name:
            context += f"\n   - Phòng: {r.room_name}"
        if r.capacity:
            context += f" (Sức chứa: {r.capacity} người)"
        
        context_parts.append(context)
    
    context_text = "\n".join(context_parts)
    
    # ----------------------------------------------------
    # BƯỚC 3: RẼ NHÁNH PROMPT NGHIÊM NGẶT THEO INTENT
    # ----------------------------------------------------
    # TRƯỜNG HỢP A: Người dùng hỏi ĐÍCH DANH một địa điểm cụ thể
    if intent == QueryIntent.SPECIFIC_SEARCH:
        prompt = f"""Bạn là trợ lý du lịch thông minh tại Đà Nẵng. Người dùng đang muốn tra cứu thông tin CHI TIẾT về một địa điểm/khách sạn/nhà hàng ĐÍCH DANH.

### Câu hỏi của người dùng:
{query}

### Thông tin tham khảo trích xuất từ cơ sở dữ liệu:
{context_text if context_text else "KHÔNG CÓ DỮ LIỆU PHÙ HỢP TRONG DATABASE."}

### QUY TẮC BẮT BUỘC KHI TRẢ LỜI (TUÂN THỦ TUYỆT ĐỐI):
1. TRẢ LỜI ĐÚNG TRỌNG TÂM: Chỉ tập trung cung cấp đầy đủ thông tin (Địa chỉ, Đánh giá, Giá cả, Mô tả) của ĐÚNG địa điểm/khách sạn/nhà hàng được yêu cầu trong câu hỏi.
2. TUYỆT ĐỐI KHÔNG GỢI Ý LUNG TUNG: Không liệt kê thêm các khách sạn/nhà hàng đối thủ khác mang tính chất so sánh hay đề xuất danh sách "Top 3". Người dùng chỉ cần thông tin của địa điểm họ hỏi.
3. KHÔNG ẢO GIÁC: Chỉ dùng thông tin trong mục "Thông tin tham khảo". Không tự bịa ra thông tin nằm ngoài ngữ cảnh.
4. XỬ LÝ KHI TRỐNG DỮ LIỆU: Nếu mục "Thông tin tham khảo" trống, hãy trả lời lịch sự: "Hiện hệ thống không tìm thấy thông tin của [tên địa điểm] trong cơ sở dữ liệu."

### Trả lời:"""

    # TRƯỜNG HỢP B: Các câu hỏi tìm kiếm theo tiêu chí, ngân sách, danh sách bình thường
    else:
        prompt = f"""Bạn là trợ lý du lịch thông minh tại Đà Nẵng. Hãy trả lời câu hỏi của người dùng dựa trên thông tin có sẵn.

### Câu hỏi của người dùng:
{query}

### Thông tin tham khảo trích xuất từ cơ sở dữ liệu:
{context_text if context_text else "KHÔNG CÓ DỮ LIỆU PHÙ HỢP TRONG DATABASE."}

### QUY TẮC BẮT BUỘC KHI TRẢ LỜI (TUÂN THỦ TUYỆT ĐỐI):
1. KHÔNG ẢO GIÁC: Chỉ được sử dụng thông tin được cung cấp trong mục "Thông tin tham khảo". KHÔNG TỰ Ý BỊA RA tên địa điểm, giá cả hoặc địa chỉ nằm ngoài ngữ cảnh trên.
2. XỬ LÝ KHI THIẾU THÔNG TIN: Nếu mục "Thông tin tham khảo" bị trống hoặc ghi "KHÔNG CÓ DỮ LIỆU PHÙ HỢP", hãy trả lời lịch sự rằng: "Hiện tại hệ thống không tìm thấy kết quả nào thỏa mãn chính xác các tiêu chí của bạn trong cơ sở dữ liệu." Tuyệt đối không tự suy diễn thông tin bên ngoài.
3. ĐỀ XUẤT PHÙ HỢP: Nếu dữ liệu có sẵn, hãy đề xuất tối đa TOP 3 lựa chọn phù hợp nhất, kèm theo Đánh giá (Rating), Giá cả và Địa chỉ rõ ràng.
4. Giữ phong thái chuyên nghiệp, thân thiện và phản hồi hoàn toàn bằng tiếng Việt.

### Trả lời:"""

    # ----------------------------------------------------
    # BƯỚC 4: RE-FORMAT MESSAGES & MODEL GENERATION
    # ----------------------------------------------------
    messages = [
        {"role": "system", "content": "Bạn là trợ lý du lịch Đà Nẵng chuyên nghiệp, thân thiện."},
        {"role": "user", "content": prompt},
    ]

    # Apply chat template
    if hasattr(tokenizer, "apply_chat_template"):
        try:
            model_inputs = tokenizer.apply_chat_template(
                messages,
                add_generation_prompt=True,
                return_tensors="pt",
                return_dict=True,
            )
        except TypeError:
            input_ids = tokenizer.apply_chat_template(
                messages,
                add_generation_prompt=True,
                return_tensors="pt",
            )
            model_inputs = {"input_ids": input_ids}
    else:
        prompt_text = f"System: {messages[0]['content']}\nUser: {messages[1]['content']}\nAssistant:"
        model_inputs = tokenizer(prompt_text, return_tensors="pt")

    # Move to device
    if hasattr(model_inputs, "to"):
        model_inputs = model_inputs.to(model.device)
    else:
        model_inputs = {
            k: (v.to(model.device) if hasattr(v, "to") else v)
            for k, v in model_inputs.items()
        }

    prompt_len = model_inputs["input_ids"].shape[-1]

    # Generate
    with torch.no_grad():
        output_ids = model.generate(
            **model_inputs,
            max_new_tokens=max_new_tokens,
            do_sample=temperature > 0,
            temperature=temperature,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    gen_tokens = output_ids[0][prompt_len:]
    answer = tokenizer.decode(gen_tokens, skip_special_tokens=True).strip()
    
    return answer

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Local LLM ready: Qwen/Qwen2.5-3B-Instruct


In [6]:
# ============================================
# CELL 5.5: LLM Query Analyzer (NÂNG CẤP FEW-SHOT)
# ============================================
import re
import json
import torch

class LLMQueryAnalyzer:
    def __init__(self, model, tokenizer):
        self.model = model
        self.tokenizer = tokenizer
        
    def _clean_price(self, value):
        """Xử lý và chuẩn hóa mọi định dạng số tiền từ LLM về VND"""
        if value is None or value == "" or str(value).lower() == "null":
            return None
        try:
            # Chuyển về chuỗi chữ thường, xóa dấu phân cách hàng nghìn (dấu phẩy, dấu chấm)
            val_str = str(value).lower().strip()
            val_str = val_str.replace(",", "").replace(".", "")
            
            # Trích xuất tất cả các chữ số liên tiếp đầu tiên tìm thấy
            digits_match = re.search(r'\d+', val_str)
            if not digits_match:
                return None
                
            val = int(digits_match.group(0))
            
            # Bắt các từ khóa hàng triệu, hàng nghìn (củ, triệu, tr, k)
            if "triệu" in val_str or "trieu" in val_str or "củ" in val_str or "cu" in val_str or "tr" in val_str:
                # Nếu LLM trả về đúng "2000000" nhưng vẫn viết chữ "triệu" phía sau, tránh nhân đôi
                if val < 10000: 
                    val = val * 1_000_000
            elif "ngàn" in val_str or "nghin" in val_str or "k" in val_str:
                if val < 10000:
                    val = val * 1_000
                    
            # Giới hạn an toàn: Nếu số tiền quá nhỏ (< 5000) và không có hậu tố, khả năng cao LLM viết tắt (ví dụ: 200 tức là 200k)
            if val < 5000 and val > 0:
                val = val * 1_000 # Tự động đưa về nghìn đồng nếu là hàng quán ăn uống
                if val < 50000 and "hotel" in str(self).lower(): # Khách sạn thì có thể là trăm k
                    val = val * 10 
    
            # Trần bảo vệ hệ thống tránh ảo giác quá lớn
            if val > 200_000_000:
                return 200_000_000
                
            return val
        except Exception as e:
            print(f"  [DEBUG] Lỗi ép kiểu giá: {e}")
            return None

    def analyze(self, query: str) -> dict:
        """Sử dụng LLM kèm Few-shot để phân tích ngữ nghĩa chính xác cấu trúc JSON"""
        
        # System prompt định hình rõ ràng vai trò và cung cấp ví dụ chuẩn
        system_content = (
            "Bạn là một AI chuyên trích xuất dữ liệu JSON cấu trúc từ câu hỏi du lịch Đà Nẵng.\n"
            "Chỉ trả về DUY NHẤT một khối JSON. Không giải thích, không thêm text ngoài JSON.\n"
            "Cấu trúc JSON bắt buộc phải tuân theo chính xác schema sau:\n"
            "{\n"
            '  "intent": "hotel_search" | "restaurant_search" | "place_search" | "review_search" | "general",\n'
            '  "rewritten_query": "chuỗi từ khóa tìm kiếm rút gọn để tạo embedding",\n'
            '  "filters": {\n'
            '    "district": "son tra" | "hai chau" | "ngu hanh son" | "cam le" | "lien chieu" | "thanh khe" | null,\n'
            '    "min_rating": float | null,\n'
            '    "max_price": int_VND | null,\n'
            '    "min_price": int_VND | null\n'
            "  }\n"
            "}"
        )
        
        # Cung cấp ví dụ Few-shot để mô hình học cách xử lý teencode và hướng giá (đổ lại/trở lên)
        user_prompt = f"""Hãy phân tích câu hỏi người dùng sau đây dựa trên các ví dụ mẫu:

### VÍ DỤ 1:
Người dùng: "Có ks nào xịn xịn cỡ 2 củ ở q. Hải Châu ko shop?"
Trả về JSON:
{{
  "intent": "hotel_search",
  "rewritten_query": "khách sạn xịn cao cấp Hải Châu",
  "filters": {{
    "district": "hai chau",
    "min_rating": null,
    "max_price": 2000000,
    "min_price": null
  }}
}}

### VÍ DỤ 2:
Người dùng: "Cho mình xin vài địa chỉ ăn hải sản ngon mà giá khoảng 1 triệu đổ lại nhé"
Trả về JSON:
{{
  "intent": "restaurant_search",
  "rewritten_query": "nhà hàng hải sản ngon",
  "filters": {{
    "district": null,
    "min_rating": null,
    "max_price": 1000000,
    "min_price": null
  }}
}}

### VÍ DỤ 3:
Người dùng: "quán ăn nào ở ngũ hành sơn được đánh giá trên 4.5 sao"
Trả về JSON:
{{
  "intent": "restaurant_search",
  "rewritten_query": "quán ăn ngon ngũ hành sơn",
  "filters": {{
    "district": "ngu hanh son",
    "min_rating": 4.5,
    "max_price": null,
    "min_price": null
  }}
}}

### VÍ DỤ 4:
Người dùng: "Tôi muốn biết thông tin về khách sạn Sala Danang Beach Hotel"
Trả về JSON:
{{
  "intent": "specific_search",
  "rewritten_query": "Sala Danang Beach Hotel",
  "filters": {{
    "district": null,
    "min_rating": null,
    "max_price": null,
    "min_price": null
  }}
}}

### BÀI TẬP THỰC TẾ:
Người dùng: "{query}"
Trả về JSON:"""

        messages = [
            {"role": "system", "content": system_content},
            {"role": "user", "content": user_prompt}
        ]

        try:
            if hasattr(self.tokenizer, "apply_chat_template"):
                model_inputs = self.tokenizer.apply_chat_template(
                    messages, 
                    add_generation_prompt=True, 
                    return_tensors="pt", 
                    return_dict=True
                )
            else:
                prompt_text = f"System: {messages[0]['content']}\nUser: {messages[1]['content']}\nAssistant:"
                model_inputs = self.tokenizer(prompt_text, return_tensors="pt", return_dict=True)
                
            model_inputs = {k: v.to(self.model.device) for k, v in model_inputs.items()}
            prompt_len = model_inputs["input_ids"].shape[-1]
            
            with torch.no_grad():
                output_ids = self.model.generate(
                    **model_inputs,
                    max_new_tokens=256,
                    do_sample=False, 
                    pad_token_id=self.tokenizer.eos_token_id,
                    eos_token_id=self.tokenizer.eos_token_id
                )
            
            gen_text = self.tokenizer.decode(output_ids[0][prompt_len:], skip_special_tokens=True).strip()
            
            # Khử nhiễu văn bản bọc ngoài JSON
            json_match = re.search(r'\{.*\}', gen_text, re.DOTALL)
            if json_match:
                parsed_json = json.loads(json_match.group(0))
            else:
                parsed_json = json.loads(gen_text)
                
            # Đảm bảo dọn dẹp và giữ đúng cấu trúc filter mong muốn
            raw_filters = parsed_json.get("filters", {})
            if not isinstance(raw_filters, dict):
                raw_filters = {}
                
            cleaned_filters = {
                "district": raw_filters.get("district"),
                "min_rating": raw_filters.get("min_rating"),
                "max_price": self._clean_price(raw_filters.get("max_price")),
                "min_price": self._clean_price(raw_filters.get("min_price"))
            }
            
            # Chuẩn hóa district text đầu ra
            if cleaned_filters["district"]:
                cleaned_filters["district"] = str(cleaned_filters["district"]).lower().strip()
            
            try:
                intent_enum = QueryIntent(parsed_json.get("intent", "general"))
            except ValueError:
                intent_enum = QueryIntent.GENERAL
                
            return {
                "intent": intent_enum,
                "rewritten_query": parsed_json.get("rewritten_query", query),
                "filters": cleaned_filters,
                "source": "LLM"
            }
            
        except Exception as e:
            print(f"  [DEBUG] LLM Parsing Error: {e}. Áp dụng Fallback an toàn (Mặc định).")
            # Khi LLM lỗi và không còn hàm rule-based, ta trả về giá trị mặc định an toàn:
            # - Intent: GENERAL (tìm kiếm toàn bộ)
            # - Rewritten query: Dùng luôn câu gốc của người dùng
            # - Filters: Rỗng (không lọc gì cả)
            return {
                "intent": QueryIntent.GENERAL, 
                "rewritten_query": query,
                "filters": {
                    "district": None,
                    "min_rating": None,
                    "max_price": None,
                    "min_price": None
                },
                "source": "LLM_Fallback"
            }

# Khởi động lại analyzer
query_analyzer = LLMQueryAnalyzer(model, tokenizer)
print("LLM Query Analyzer nâng cấp Few-shot đã sẵn sàng!")

LLM Query Analyzer nâng cấp Few-shot đã sẵn sàng!


In [7]:
# ============================================
# CELL 6: Main RAG Pipeline (Cập nhật tích hợp LLM)
# ============================================

class RAGPipeline:
    """Complete RAG pipeline orchestrator with LLM Query Analyzer"""
    
    def __init__(self):
        self.encoder = encoder
        self.client = client
        self.analyzer = query_analyzer # Tích hợp analyzer vào pipeline
        self.default_top_k = 5
        self.default_max_results = 15
    
    def search(
        self,
        query: str,
        top_k: int = 5,
        score_threshold: float = 0.3
    ) -> Tuple[List[SearchResult], QueryIntent, dict]:
        
        # 1. Phân tích query bằng LLM
        analysis_result = self.analyzer.analyze(query)
        intent = analysis_result["intent"]
        filters = analysis_result["filters"]
        rewritten_query = analysis_result["rewritten_query"]
        source = analysis_result["source"]
        
        # 2. Truy xuất bằng rewritten_query thay vì query gốc
        results = retrieve_by_intent(
            query=rewritten_query, 
            intent=intent,
            top_k_per_collection=top_k,
            max_total_results=self.default_max_results,
            filters=filters
        )
        
        results = [r for r in results if r.score >= score_threshold]
        
        # Trả về cả context phân tích để in ra màn hình
        context = {
            "intent": intent,
            "rewritten_query": rewritten_query,
            "filters": filters,
            "source": source
        }
        return results, intent, context
    
    def answer(
        self,
        query: str,
        top_k: int = 5,
        max_tokens: int = 512,
        temperature: float = 0.2
    ) -> Dict[str, Any]:
        """Get answer with context and deduplicated results"""
        
        # 1. Chạy pipeline tìm kiếm gốc
        results, intent, context = self.search(query, top_k=top_k)
        
        # 2. KHỬ TRÙNG LẶP THỰC THỂ (Deduplication)
        # Đảm bảo các thực thể trùng tên từ nhiều nguồn cào không xuất hiện đè lên nhau
        deduplicated_results = []
        seen_names = set()
        for r in results:
            name_key = r.get_display_name().lower().strip()
            if name_key not in seen_names:
                seen_names.add(name_key)
                deduplicated_results.append(r)
            else:
                # Nếu gặp bản ghi trùng tên, ưu tiên giữ lại bản ghi có thông tin giá chi tiết hơn
                existing_idx = next(i for i, item in enumerate(deduplicated_results) if item.get_display_name().lower().strip() == name_key)
                if not deduplicated_results[existing_idx].min_price and r.min_price:
                    deduplicated_results[existing_idx] = r
        
        # 3. Tạo câu trả lời tự nhiên từ LLM (Dùng danh sách sạch đã khử trùng)
        answer = generate_answer(query, deduplicated_results, max_new_tokens=max_tokens, temperature=temperature, intent=intent)
        
        # 4. Định dạng lại danh sách hiển thị phía dưới màn hình (📋 Kết quả tìm thấy phù hợp)
        filters = context["filters"]
        formatted_results = []
        for r in deduplicated_results[:10]: # Hiển thị tối đa 10 thực thể độc bản
            if filters and filters.get("max_price"):
                if r.min_price and r.min_price > filters["max_price"]:
                    continue
            
            formatted_results.append({
                "name": r.get_display_name(),
                "type": r.collection,
                "district": r.district,
                "rating": r.get_rating_display(),
                "price": r.get_price_display(),
                "address": r.address,
                "score": round(r.score, 3)
            })
            
        return {
            "query": query,
            "analysis_context": context,
            "answer": answer,
            "results": formatted_results,
            "total_results": len(deduplicated_results)
        }

In [8]:
# ============================================
# CELL 7: Demo RAG với LLM Query Analyzer
# ============================================

print("=" * 80)
print("RAG SYSTEM DEMO (INTEGRATED LLM QUERY ANALYZER)")
print("=" * 80)

# Khởi tạo RAG Pipeline
print("🔄 Khởi tạo RAG Pipeline...")
pipeline = RAGPipeline()
print("✅ Pipeline initialized successfully!")

# Những câu hỏi phức tạp để test độ "bén" của LLM so với Regex
test_queries = [
       ("🏨 Khách sạn", 
     "Gợi ý khách sạn ở quận Sơn Trà có đánh giá tốt trên 8.0"),

    ("🏨 Khách sạn giá rẻ", 
     "Khách sạn nào ở Đà Nẵng có giá dưới 1 triệu đồng?"),

    ("🏨 Khách sạn cao cấp", 
     "Khách sạn 5 sao nào ở Đà Nẵng được đánh giá cao?"),

    ("🏨 View biển", 
     "Khách sạn nào gần biển Mỹ Khê và có view đẹp?"),

    ("🏨 Gia đình", 
     "Khách sạn phù hợp cho gia đình 4 người ở Đà Nẵng"),

    ("🏨 Cặp đôi", 
     "Resort nào phù hợp cho cặp đôi nghỉ dưỡng?"),

    ("🏨 Hồ bơi", 
     "Khách sạn nào có hồ bơi vô cực đẹp?"),

    ("🏨 Gần trung tâm", 
     "Khách sạn gần cầu Rồng và trung tâm thành phố"),

    ("🏨 Review", 
     "Review về khách sạn Le Sands Oceanfront Danang Hotel "),

    ("🏨 Giá phòng", 
     "Giá phòng khách sạn ở Đà Nẵng khoảng bao nhiêu?"),

    ("🏨 Phòng lớn", 
     "Khách sạn nào có phòng gia đình sức chứa 4-5 người?"),

    ("🏨 Buffet sáng", 
     "Khách sạn nào có buffet sáng ngon?"),

    ("🏨 Check-in đẹp", 
     "Khách sạn nào có không gian check-in sống ảo đẹp?"),

    ("🏨 Nghỉ dưỡng", 
     "Gợi ý resort yên tĩnh để nghỉ dưỡng cuối tuần"),

    # ==================================================
    # NHÀ HÀNG
    # ==================================================

    ("🍽️ Nhà hàng", 
     "Nhà hàng nào ở Hải Châu có không gian đẹp và đồ ăn ngon?"),

    ("🍽️ Hải sản", 
     "Gợi ý nhà hàng hải sản ngon ở Đà Nẵng"),

    ("🍽️ Mì Quảng", 
     "Quán mì Quảng nổi tiếng ở Đà Nẵng"),

    ("🍽️ Bún chả cá", 
     "Ăn bún chả cá ngon ở đâu tại Đà Nẵng?"),

    ("🍽️ Đặc sản", 
     "Nhà hàng nào bán đặc sản Đà Nẵng ngon?"),

    ("🍽️ Buffet", 
     "Buffet nướng hải sản nào ngon và giá hợp lý?"),

    ("🍽️ Sang trọng", 
     "Nhà hàng sang trọng thích hợp tiếp khách"),

    ("🍽️ Giá rẻ", 
     "Quán ăn ngon bổ rẻ cho sinh viên ở Đà Nẵng"),

    ("🍽️ Gia đình", 
     "Nhà hàng phù hợp cho gia đình có trẻ em"),

    ("🍽️ Mở khuya", 
     "Quán ăn nào mở khuya ở Đà Nẵng?"),

 
    # ==================================================
    # ĐỊA ĐIỂM DU LỊCH
    # ==================================================

    ("📍 Địa điểm", 
     "Địa điểm du lịch nổi tiếng nào ở bán đảo Sơn Trà?"),

    ("📍 Check-in", 
     "Địa điểm check-in đẹp ở Đà Nẵng"),

    ("📍 Biển", 
     "Bãi biển nào đẹp nhất ở Đà Nẵng?"),

    ("📍 Ban đêm", 
     "Buổi tối ở Đà Nẵng nên đi đâu chơi?"),

    ("📍 Thiên nhiên", 
     "Địa điểm thiên nhiên đẹp và yên bình"),

    ("📍 Văn hóa", 
     "Các địa điểm mang nét văn hóa và lịch sử ở Đà Nẵng"),

    ("📍 Gia đình", 
     "Địa điểm vui chơi phù hợp cho gia đình có trẻ em"),

    ("📍 Miễn phí", 
     "Địa điểm tham quan miễn phí ở Đà Nẵng"),


    ("📍 Cuối tuần", 
     "Cuối tuần nên đi đâu ở Đà Nẵng?"),

    ("📍 Chụp ảnh", 
     "Địa điểm chụp ảnh đẹp lúc hoàng hôn"),

  

    # ==================================================
    # Ý ĐỊNH PHỨC TẠP
    # ==================================================


    ("🧠 Kết hợp", 
     "Gợi ý khách sạn gần biển và có nhà hàng hải sản ngon xung quanh"),

    ("🧠 Theo nhu cầu", 
     "Tôi thích nơi yên tĩnh, ít đông người, nên ở đâu?"),

    # --- NHÓM 5: ĐỊA ĐIỂM / TRUY VẤN CHUNG (Place & General Intent) ---
    ("🏛️ Địa điểm / Văn hóa", 
     "Buổi tối ở Đà Nẵng thì nên đi tham quan chỗ nào mang tính lịch sử văn hóa nhỉ?"),
     
    ("❓ Câu hỏi chung", 
     "Mùa hè đi chơi đâu tại Đà Nẵng?")
]

for category, query in test_queries:
    print(f"\n{'='*80}")
    print(f"[{category}]")
    print(f"Câu hỏi gốc: {query}")
    print("-" * 80)
    
    try:
        result = pipeline.answer(query, top_k=5, temperature=0.2)
        
        analysis = result['analysis_context']
        print(f"⚙️ Analyzer Source: {analysis['source']}")
        print(f"🎯 Intent        : {analysis['intent'].value}")
        print(f"✍️ Rewritten     : {analysis['rewritten_query']}")
        print(f"🔍 Filters       : {analysis['filters']}")
        print("-" * 80)
        
        print(f"\n🤖 Trả lời:\n{result['answer']}")
        print(f"\n📋 Kết quả tìm thấy phù hợp:")
        for i, r in enumerate(result['results'][:5], 1):
            print(f"\n  {i}. {r['name']}")
            print(f"     - Loại: {r['type']}")
            if r.get('district'): print(f"     - Quận: {r['district']}")
            if r.get('rating'): print(f"     - Đánh giá: {r['rating']}")
            if r.get('price') and r['price'] != "Không có thông tin giá": 
                print(f"     - Giá: {r['price']}")
                
    except Exception as e:
        print(f"❌ Error: {e}")
        import traceback
        traceback.print_exc()

print("\n" + "="*80)
print("DEMO HOÀN TẤT")
print("="*80)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


RAG SYSTEM DEMO (INTEGRATED LLM QUERY ANALYZER)
🔄 Khởi tạo RAG Pipeline...
✅ Pipeline initialized successfully!

[🏨 Khách sạn]
Câu hỏi gốc: Gợi ý khách sạn ở quận Sơn Trà có đánh giá tốt trên 8.0
--------------------------------------------------------------------------------
⚙️ Analyzer Source: LLM
🎯 Intent        : hotel_search
✍️ Rewritten     : khách sạn Sơn Trà đánh giá cao
🔍 Filters       : {'district': 'son tra', 'min_rating': 8.0, 'max_price': None, 'min_price': None}
--------------------------------------------------------------------------------

🤖 Trả lời:
Dựa trên thông tin mà tôi có, bạn có thể cân nhắc đến một số khách sạn sau:

1. **Hotel Le Bouton**
   - **Đánh giá:** 9.1/10
   - **Giá:** 1,042,801 - 1,215,035 VND
   - **Địa chỉ:** 12 Trần Đình Đàn, Sơn Trà, Đà Nẵng

2. **Son Tra Resort & Spa Danang**
   - **Đánh giá:** 8.0/10
   - **Giá:** 2,997,432 - 6,280,334 VND
   - **Địa chỉ:** Nam Son Tra Beach, Đà Nẵng

3. **Sala Danang Beach Hotel**
   - **Đánh giá:** 9.4/10
  